# PharmaDemand — DataFrame → Features + IF → Parquet



In [1]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.features.build_features import build_features_dataframe, feature_columns
from src.repository.parquet_repository import ParquetRepository


In [2]:
# Passo 1: uma única leitura da base original
df_salesdaily = pd.read_csv(ROOT / "data/raw/salesdaily.csv")
df_salesdaily.head()


,datum,M01AB,M01AE,N02BA,N02BE,N05B,N05C,R03,R06,Year,Month,Hour,Weekday Name
0,1/2/2014,0.0,3.67,3.4,32.40,7.0,0.0,0.0,2.0,2014,1,248,Thursday
1,1/3/2014,8.0,4.00,4.4,50.60,16.0,0.0,20.0,4.0,2014,1,276,Friday
2,1/4/2014,2.0,1.00,6.5,61.85,10.0,0.0,9.0,1.0,2014,1,276,Saturday
3,1/5/2014,4.0,3.00,7.0,41.10,8.0,0.0,3.0,0.0,2014,1,276,Sunday
4,1/6/2014,5.0,1.00,4.5,21.70,16.0,2.0,6.0,2.0,2014,1,276,Monday


In [3]:
# Passo 2: computa todas as colunas e as duas do Isolation Forest EM MEMÓRIA
df_features, if_artifacts = build_features_dataframe(
    df_salesdaily, category="N02BE", horizons=(7, 15),
    test_months=6, contamination=0.03,
)
print("Shape:", df_features.shape)
#display(df_features[["date", "horizon_days", "quantity", "anomaly_score", "is_anomaly"]].tail(50))
display (df_features[df_features["is_anomaly"] == 1])


Shape: (4078, 46)


,date,category,horizon_days,forecast_origin,split,quantity,year,month,quarter,weekday,...,qty_roll_std_56,qty_mean_ratio_7_28,qty_mean_delta_7_28,qty_last_ratio_28,qty_last_zscore_28,qty_cv_28,qty_zero_ratio_28,qty_mean_ratio_14_56,anomaly_score,is_anomaly
304,2015-01-04,N02BE,7,2014-12-28,train,60.100,2015,1,1,6,...,12.114994,1.279794,8.829286,2.547817,3.135497,0.493643,0.035714,1.245661,0.583289,1
597,2015-10-24,N02BE,7,2015-10-17,train,67.400,2015,10,4,5,...,15.841053,1.481120,20.319571,1.759252,1.829546,0.414995,0.000000,1.539841,0.596380,1
667,2016-01-02,N02BE,7,2015-12-26,train,51.200,2016,1,1,5,...,16.188173,1.192344,7.133893,2.351085,3.225299,0.418902,0.035714,0.932911,0.630807,1
673,2016-01-08,N02BE,7,2016-01-01,train,39.200,2016,1,1,4,...,17.112373,1.384707,15.262536,0.000000,-2.084699,0.479686,0.071429,1.109250,0.713179,1
679,2016-01-14,N02BE,7,2016-01-07,train,69.000,2016,1,1,3,...,17.531161,0.901479,-3.971857,0.000000,-1.950740,0.512626,0.107143,1.222353,0.594627,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3841,2019-02-14,N02BE,15,2019-01-30,train,54.500,2019,2,1,3,...,22.509126,1.060345,3.365571,0.937964,-0.159016,0.390123,0.035714,1.341897,0.544904,1
3842,2019-02-15,N02BE,15,2019-01-31,train,52.050,2019,2,1,4,...,21.722001,1.084757,4.787000,1.101294,0.264455,0.383030,0.035714,1.320366,0.578657,1
3843,2019-02-16,N02BE,15,2019-02-01,train,36.400,2019,2,1,5,...,21.750646,1.112454,6.283429,0.696192,-0.775911,0.391551,0.035714,1.341421,0.574701,1
3844,2019-02-17,N02BE,15,2019-02-02,train,15.162,2019,2,1,6,...,20.910384,1.025716,1.438286,1.148984,0.380538,0.391508,0.035714,1.290146,0.652156,1


In [4]:
# Passo 3: persistir apenas o resultado final em arquivo .parquet
repo = ParquetRepository(base_path=ROOT / "data")
repo.save(df_features, "processed/salesdaily_features_N02BE.parquet")
print(df_features.groupby(["horizon_days", "split"]).size())


[OK] Arquivo salvo em: d:\PROJETOS\git_repo\PharmaDemand-ml\data\processed\salesdaily_features_N02BE.parquet
horizon_days  split
7             test      183
              train    1860
15            test      183
              train    1852
dtype: int64


In [5]:
# Exemplo futuro de leitura para o LightGBM, com h=15
df = repo.load("processed/salesdaily_features_N02BE.parquet")
df_15 = df[df["horizon_days"] == 15]
X_train = df_15.loc[df_15["split"] == "train", feature_columns(df_15)]
y_train = df_15.loc[df_15["split"] == "train", "quantity"]
print(X_train.shape, y_train.shape)


[OK] Arquivo carregado: d:\PROJETOS\git_repo\PharmaDemand-ml\data\processed\salesdaily_features_N02BE.parquet
(1852, 40) (1852,)
